In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from statsmodels.stats.multitest import multipletests

In [17]:
#Chi-squared for cluster membership vs. surgery before/after
group_annotations = pd.read_csv('../data/processed/group_annotation.csv', index_col=0)
top_variable_expression=pd.read_csv('../results/top_5000_variable_expression.csv', index_col=0)
group_annotations = group_annotations.loc[top_variable_expression.T.index]

k=3
data_for_clustering = top_variable_expression.T
agglo = AgglomerativeClustering(n_clusters=k, compute_distances=True)
agglo_cluster_labels = agglo.fit_predict(data_for_clustering)

k=3
data_for_clustering = top_variable_expression.T
kmeans = KMeans(n_clusters=k, random_state=27)
kmeans_labels = kmeans.fit_predict(data_for_clustering)





In [32]:
def chi_squared_clusters_vs_surgery(cluster_labels):
   contingency_table = pd.crosstab(cluster_labels, group_annotations['Group']=='T0')
   chi2, p, _, _ = chi2_contingency(contingency_table)
   #res = pd.DataFrame({'Genes 1': n1, 'Genes 2': n2, 'Chi2 Statistic': chi2, 'p-value': p})
  
   return chi2, p

In [37]:
chi2, p = chi_squared_clusters_vs_surgery(agglo_cluster_labels)
print(f"Agglomerative: chi2={chi2:.2f}, p-value={p:.6f}")

chi2, p = chi_squared_clusters_vs_surgery(kmeans_labels)
print(f"KMeans: chi2={chi2:.2f}, p-value={p:.6f}")

Agglomerative: chi2=17.50, p-value=0.000159
KMeans: chi2=24.41, p-value=0.000005


Adjustment for Multiple Hypothesis Testing

In [38]:

# Example: run for multiple clustering results
results = []
clusterings = {
    'Agglomerative': agglo_cluster_labels,
    'KMeans': kmeans_labels,
}

for name, labels in clusterings.items():
    chi2, p = chi_squared_clusters_vs_surgery(labels)
    results.append({'Method': name, 'Chi2': chi2, 'p-value': p})

results_df = pd.DataFrame(results)

# Adjust p-values for multiple testing (Benjamini-Hochberg FDR)
results_df['adj_p-value'] = multipletests(results_df['p-value'], method='fdr_bh')[1]

print(results_df)

          Method       Chi2   p-value  adj_p-value
0  Agglomerative  17.497277  0.000159     0.000159
1         KMeans  24.405280  0.000005     0.000010
